<a href="https://colab.research.google.com/github/Taheri1400/TensorFlow/blob/main/7_eager_vs_graph_execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
اجرای بی‌تابانه و اجرای گرافی‌‌
</font>
</h1>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مقایسه
</font>
</h2>

In [ ]:
def my_decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@my_decorator
def quera():
    print("Quera!")

quera()

Before
Quera!
After


In [ ]:
import tensorflow as tf

a = tf.constant(2)
print(a)

b = tf.constant(3)
print(b)

c = a + b

print("Result using eager execution:", c.numpy())

tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
Result using eager execution: 5


In [ ]:
import tensorflow as tf

# Graph execution decorator
@tf.function
def add_numbers():

    a = tf.constant(2)
    print(a)

    b = tf.constant(3)
    print(b)

    c = a + b
    print(c)

    return a, b, c

# execute the graph
a, b, c = add_numbers()

print('-'*50)
print(a.numpy())
print(b.numpy())
print("Result using graph execution:", c.numpy())

Tensor("Const:0", shape=(), dtype=int32)
Tensor("Const_1:0", shape=(), dtype=int32)
Tensor("add:0", shape=(), dtype=int32)
--------------------------------------------------
2
3
Result using graph execution: 5


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.data import AUTOTUNE

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مدل‌سازی
</font>
</h2>

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_labels = tf.one_hot(train_labels, 10)
test_labels = tf.one_hot(test_labels, 10)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))

In [ ]:
# Reshape the images and normalize the pixel values
train_dataset = train_dataset.map(lambda image, label: (tf.reshape(tf.cast(image, tf.float32) / 255.0, [784]), label))
test_dataset = test_dataset.map(lambda image, label: (tf.reshape(tf.cast(image, tf.float32) / 255.0, [784]), label))

In [ ]:
train_dataset = (train_dataset
                 .shuffle(512)
                 .cache()
                 .batch(32)
                 .prefetch(AUTOTUNE)
                )

test_dataset = (test_dataset
                 .cache()
                 .batch(32)
                 .prefetch(AUTOTUNE)
                )

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تابع آموزش روی یک دسته
</font>
</h3>

In [ ]:
# Graph execution decorator
@tf.function
def fit_one_batch(X, y, model, loss_fn, optimizer):
  with tf.GradientTape() as tape:
    y_prime = model(X)
    loss = loss_fn(y, y_prime)

  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))

  return (y_prime, loss)

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
تابع آموزش روی یک دور
</font>
</h3>

In [ ]:
def fit(dataset: tf.data.Dataset, model: tf.keras.Model, loss_fn: tf.keras.losses.Loss,
optimizer: tf.optimizers.Optimizer) -> None:
  loss_sum = 0
  correct_item_count = 0
  current_item_count = 0
  print_every = 100
  batch_loss = 0
  batch_index = 0

  for (X, y) in dataset:
    (y_prime, loss) = fit_one_batch(X, y, model, loss_fn, optimizer)

    y = tf.cast(y, tf.int64)
    correct_item_count += tf.reduce_sum(tf.cast(tf.math.argmax(y_prime, axis=1) == tf.math.argmax(y, axis=1), tf.int32)).numpy()

    batch_loss = loss.numpy()
    loss_sum += batch_loss
    current_item_count += len(X)
    batch_index += 1

    # If the remainder is zero, it means that 'batch_index' is a multiple of 'print_every'
    if ((batch_index) % print_every == 0):
      batch_accuracy = correct_item_count / current_item_count * print_every
      print(f'[Batch {batch_index:>3d} - {current_item_count:>5d} items] accuracy: {batch_accuracy:>0.1f}%, loss: {batch_loss:>7f}')

  batch_accuracy = correct_item_count / current_item_count * print_every
  print(f'[Batch {batch_index:>3d} - {current_item_count:>5d} items] accuracy: {batch_accuracy:>0.1f}%, loss: {batch_loss:>7f}')

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
ساختار مدل
</font>
</h3>

In [ ]:
class QDense(tf.Module):
    def __init__(self, in_features, out_features, activation=tf.identity, name=None):
        super().__init__(name=name)

        # Xavier/Glorot initialization for weights
        initializer = tf.initializers.GlorotUniform()
        self.w = tf.Variable(initializer(shape=[in_features, out_features]), name='w')

        self.b = tf.Variable(tf.zeros([out_features]), name='b')

        self.activation = activation

    def __call__(self, x):
        y = tf.matmul(x, self.w) + self.b
        y = self.activation(y)
        return y

In [ ]:
class QSequentialModel(tf.Module):
  def __init__(self, name=None):
    super().__init__(name=name)

    self.dense_1 = QDense(in_features=784, out_features=20, activation=tf.nn.relu)
    self.dense_2 = QDense(in_features=20, out_features=10, activation=tf.nn.softmax)

  def __call__(self, x):
    x = self.dense_1(x)
    return self.dense_2(x)

<h3 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش کامل
</font>
</h3>

In [ ]:
model = QSequentialModel(name="Quera")

loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.optimizers.SGD(0.1)

print('\nFitting:')
for epoch in range(10):
  print(f'\nEpoch {epoch + 1}\n-------------------------------')
  fit(train_dataset, model, loss_fn, optimizer)


Fitting:

Epoch 1
-------------------------------
[Batch 100 -  3200 items] accuracy: 69.8%, loss: 0.356951
[Batch 200 -  6400 items] accuracy: 78.5%, loss: 0.411262
[Batch 300 -  9600 items] accuracy: 81.6%, loss: 0.245579
[Batch 400 - 12800 items] accuracy: 83.4%, loss: 0.440167
[Batch 500 - 16000 items] accuracy: 84.5%, loss: 0.324529
[Batch 600 - 19200 items] accuracy: 85.4%, loss: 0.299646
[Batch 700 - 22400 items] accuracy: 86.3%, loss: 0.226334
[Batch 800 - 25600 items] accuracy: 87.0%, loss: 0.404219
[Batch 900 - 28800 items] accuracy: 87.5%, loss: 0.464813
[Batch 1000 - 32000 items] accuracy: 87.8%, loss: 0.302571
[Batch 1100 - 35200 items] accuracy: 88.2%, loss: 0.288559
[Batch 1200 - 38400 items] accuracy: 88.5%, loss: 0.284665
[Batch 1300 - 41600 items] accuracy: 88.8%, loss: 0.312627
[Batch 1400 - 44800 items] accuracy: 89.0%, loss: 0.123330
[Batch 1500 - 48000 items] accuracy: 89.2%, loss: 0.180149
[Batch 1600 - 51200 items] accuracy: 89.4%, loss: 0.155280
[Batch 1700 - 